# SpaNCy-Shift Explorer

Two-stage batch normalization for CyCIF multiplexed imaging.

**Stage 1 (analytic):** Per-marker shift toward KL-medoid reference sample → `normalized_base`.
Bimodal markers get separate neg/pos peak shifts. kBET ≈ 0.631 (matches UniFORM) with no DL.

**Stage 2 (deep learning):** `GNNStage2` (GATv2 + GRL + MMD) trained on Stage 1 output
→ `normalized`. Spatial context improves 20D covariance structure (kBET). Bimodal markers
masked from MMD — histograms for those markers are unchanged from Stage 1.

**Sections:**
0. Colab Setup
1. Load & Inspect Data
2. Bimodal Marker Detection
2b. Reference Sample Selection
3. Training (Stage 2)
4. Inference & Histogram Inspection
5. Batch adj-R² Diagnostics
6. Positive Population Preservation Check
7. Histogram Comparison
8. kBET Evaluation

## 0. Colab Setup

Run first on Google Colab. Installs dependencies and uploads `spancy_shift.py`.

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q anndata scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload spancy_shift.py
from google.colab import files
uploaded = files.upload()

import os
assert os.path.exists('spancy_shift.py'), 'spancy_shift.py not found — upload it first'
print(f"spancy_shift.py uploaded ({os.path.getsize('spancy_shift.py'):,} bytes)")

In [ ]:
import importlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

import spancy_shift
from spancy_shift import (
    load_adata, log1p_scale,
    detect_bimodal_markers, find_best_sample_per_marker,
    shift_normalize_per_marker,
    GNNStage2, AdjacencyIndex, train,
    normalize_adata,
    positive_population_table, summarize_positive_population,
    per_marker_batch_r2,
)

# After re-uploading spancy_shift.py:
# importlib.reload(spancy_shift); from spancy_shift import *

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Download PRAD dataset
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load & Inspect Data

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")
print(f"\nobs columns: {list(adata.obs.columns)}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)

In [ ]:
# Per-batch mean intensity overview
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].boxplot([X_raw[:, i] for i in range(X_raw.shape[1])], labels=marker_names, vert=True)
axes[0].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[0].set_title('Raw marker distributions (all cells)')
axes[0].set_ylabel('Intensity')

for b in unique_batches:
    mask = batch_vals == b
    axes[1].plot(X_raw[mask].mean(axis=0), marker='o', markersize=3, label=str(b), alpha=0.8)
axes[1].set_xticks(range(len(marker_names)))
axes[1].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[1].set_title('Per-batch mean intensity')
axes[1].legend(fontsize=6, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Bimodal Marker Detection (preview on raw data)

Adaptive per-batch voting: a marker is bimodal if ≥50% of batches independently show ≥2
prominent histogram peaks. Used in Stage 1 (log1p space).

- **Bimodal**: ≥2 peaks in majority of batches → separate neg/pos peak shifts per sample
- **Unimodal**: 1 peak → single median shift per sample

This cell previews bimodal detection on raw data. The actual detection runs inside
`shift_normalize_per_marker()` (log1p space) and the result is passed directly into Stage 2
— Stage 2 no longer re-detects bimodal markers on scaled data.

In [ ]:
# Preview: scale raw data for bimodal detection visualization
# Note: train() will run bimodal detection on log1p(X_base_scaled) internally
X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
print(f'log1p range: [{X_log_preview.min():.3f}, {X_log_preview.max():.3f}]')

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

print(f"\nBimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")

print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")

## 2b. Reference Sample Selection

For each marker, find the **medoid sample** — the one with lowest mean pairwise symmetric KL
divergence to all other samples in log1p space.  This sample is the normalization target for
that marker: all other samples are pulled toward it.  Using a data-driven reference (rather
than the global mean/quantile) is the key improvement over the previous spancy_shift approach.

In [ ]:
# Compute reference sample per marker (passed to train() to skip recomputation)
ref_sample_per_marker = find_best_sample_per_marker(adata)

# Show as table
ref_df = pd.DataFrame(
    [(m, s) for m, s in ref_sample_per_marker.items()],
    columns=['marker', 'reference_sample'],
)
print('Per-marker reference sample (normalization target):\n')
print(ref_df.to_string(index=False))

# How many distinct reference samples are used?
ref_counts = ref_df['reference_sample'].value_counts()
print(f'\nDistinct reference samples used: {len(ref_counts)}')
print(ref_counts.to_string())

# Heatmap: for each marker, which sample is the reference?
fig, ax = plt.subplots(figsize=(14, 3))
sample_order = sorted(adata.obs['sample_id'].unique())
ref_matrix = np.zeros((1, len(sample_order)))
for i, s in enumerate(sample_order):
    ref_matrix[0, i] = sum(v == s for v in ref_sample_per_marker.values())
ax.bar(range(len(sample_order)), ref_matrix[0], color='steelblue', alpha=0.85)
ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(sample_order, rotation=45, ha='right')
ax.set_ylabel('# markers using as reference')
ax.set_title('How often each sample is chosen as reference (medoid across markers)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize histograms with detected peaks
from scipy.ndimage import gaussian_filter1d

n_cols = 5
n_rows = int(np.ceil(len(marker_names) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

for m, mname in enumerate(marker_names):
    ax = axes[m]
    col = X_log_preview[:, m]
    lo, hi = np.percentile(col, [1, 99])
    bins = np.linspace(lo, hi, 151)
    counts, edges = np.histogram(col, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    smoothed = gaussian_filter1d(counts.astype(float), sigma=2)

    ax.fill_between(centers, smoothed, alpha=0.4,
                    color='steelblue' if marker_is_bimodal[m] else 'salmon')
    ax.plot(centers, smoothed, 'k-', linewidth=1)

    if marker_is_bimodal[m]:
        ax.axvline(thresholds[m], color='red', linestyle='--', linewidth=1.5,
                   label=f'thresh={thresholds[m]:.2f}')
        ax.legend(fontsize=6)

    label = 'BIMODAL' if marker_is_bimodal[m] else 'unimodal'
    ax.set_title(f'{mname}\n({label})', fontsize=8,
                 fontweight='bold' if marker_is_bimodal[m] else 'normal')
    ax.set_xlabel('log1p value', fontsize=7)
    ax.tick_params(labelsize=6)

for ax in axes[len(marker_names):]:
    ax.set_visible(False)

plt.suptitle('Marker Distributions — Bimodal Detection (log1p space, raw data preview)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Stage 2 Training (GNN)

Trains `GNNStage2` on Stage 1 output (`X_base`).

**Inside `train()`:**
1. Runs `shift_normalize_per_marker()` → `X_base` (Stage 1, analytic)
2. Fits `RobustScaler` on `log1p(X_base)` → `X_base_scaled`
3. Builds spatial k-NN graph from (x, y) coordinates per scene
4. Trains `GNNStage2` with:
   - `L_recon` — Huber (keep delta small, biology-preserving)
   - `L_contrast` — NT-Xent with spatial neighbors as positives
   - `L_adv` — adversarial batch discriminator with gradient reversal (GRL)

**Inference:** `X_out = X_base_scaled + hybrid_alpha * delta` → inverse_transform

**Key parameters:**
- `hybrid_alpha=0.3` — blend strength: 0=Stage 1 only, 1=full GNN residual
- `n_per_batch=512` — cells per batch per step in scene-based sampler
- `k_neighbors=15` — spatial k-NN connectivity
- `w_contrast=0.5` — spatial NT-Xent weight
- `w_adv=0.3` — adversarial loss weight (GRL lambda ramps 0→1 over training)

In [ ]:
HYBRID_ALPHA = 0.3
K_NEIGHBORS = 15
N_EPOCHS = 10  # quick test; use 50-100 for production

model, trained_scaler, trained_ref, history = train(
    adata,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    device_str=DEVICE,
    warmup_epochs=2,
    n_per_batch=512,
    k_neighbors=K_NEIGHBORS,
    hybrid_alpha=HYBRID_ALPHA,
    temperature=0.07,
    w_recon=0.1,    # soft regularizer only — MMD is the primary decoder signal
    w_contrast=0.5,
    w_adv=0.3,
    w_mmd=1.0,      # MMD on decoder output (unimodal markers only, across batches)
    mmd_samples=256,
    grl_max=1.0,
    bimodal_min_batch_frac=0.5,
    ref_sample_per_marker=ref_sample_per_marker,
)

print(f'\nTraining complete.')
print(f'  Final loss={history["loss"][-1]:.4f}  '
      f'recon={history["recon"][-1]:.4f}  '
      f'contrast={history["contrast"][-1]:.4f}  '
      f'adv={history["adv"][-1]:.4f}  '
      f'mmd={history["mmd"][-1]:.4f}')

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 6, figsize=(28, 4))

for ax, key, color, title in zip(
    axes,
    ['loss', 'recon', 'contrast', 'adv', 'mmd', 'lr'],
    ['k', 'darkorange', 'steelblue', 'firebrick', 'mediumorchid', 'purple'],
    ['Total Loss', 'Recon (Huber)', 'Contrast (NT-Xent)', 'Adversarial (GRL)', 'MMD (decoder out)', 'Learning Rate'],
):
    ax.plot(history[key], color=color, linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
    if key == 'lr':
        ax.set_yscale('log')

ax_grl = axes[3].twinx()
ax_grl.plot(history['grl_lambda'], 'g--', linewidth=1, alpha=0.6)
ax_grl.set_ylabel('GRL λ', color='green', fontsize=9)

plt.suptitle('Stage 2 Training — GNNStage2', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Inference & Output Inspection

Run inference with both affine-only and shift modes.
Use `shift_alpha` to blend between affine (0) and full shift (1).

In [ ]:
# Run both stages: Stage 1 (analytic) + Stage 2 (GNN residual)
# normalize_adata() runs Stage 1 internally and stores both layers.

adata_norm = normalize_adata(
    adata, model, trained_scaler, trained_ref,
    hybrid_alpha=HYBRID_ALPHA,
    k_neighbors=K_NEIGHBORS,
    inference_batch_size=2048,  # reduce if OOM; increase if GPU has headroom
    device_str=DEVICE,
    layer_name='normalized',
    keep_base_layer=True,
)

# Convenience arrays
X_base = np.asarray(adata_norm.layers['normalized_base'])   # Stage 1
X_norm = np.asarray(adata_norm.layers['normalized'])         # Stage 2

print(f'Stage 1 (normalized_base): min={X_base.min():.4f}  max={X_base.max():.4f}  mean={X_base.mean():.4f}')
print(f'Stage 2 (normalized):      min={X_norm.min():.4f}  max={X_norm.max():.4f}  mean={X_norm.mean():.4f}')
print(f'\nLayers: {list(adata_norm.layers.keys())}')

In [ ]:
# Delta magnitude: how much the GNN is correcting each marker
n_viz = min(5000, adata_norm.n_obs)
model.eval()
with torch.no_grad():
    X_base_viz = np.asarray(adata_norm.layers['normalized_base'][:n_viz], dtype=np.float32)
    X_sc_viz = torch.tensor(
        trained_scaler.transform(np.log1p(np.clip(X_base_viz, 0, None))).astype(np.float32)
    ).to(DEVICE)
    edge_dummy = torch.zeros((2, 0), dtype=torch.long, device=DEVICE)
    _, delta_viz, _, _ = model(X_sc_viz, edge_dummy)
    delta_np = delta_viz.cpu().numpy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.boxplot([delta_np[:, m] for m in range(len(marker_names))], labels=marker_names, vert=True)
ax.set_xticklabels(marker_names, rotation=90, fontsize=8)
ax.set_ylabel('Delta (scaled space)')
ax.set_title(f'GNN Residual Delta per Marker (first {n_viz} cells, no spatial context)')
ax.axhline(0, color='k', linewidth=0.5, alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Compare raw vs Stage 1 vs Stage 2 for unimodal markers
# Bimodal markers are masked from Stage 2 MMD and pass through unchanged from Stage 1 —
# their histograms will always match Stage 1 exactly and prove nothing about Stage 2 quality.
# Use unimodal markers where Stage 2 actually makes corrections.
unimodal_names = [marker_names[i] for i in range(len(marker_names)) if not marker_is_bimodal[i]]
plot_markers = unimodal_names[:4] if len(unimodal_names) >= 4 else unimodal_names

layers_to_show = [
    (None,                 'Raw'),
    ('normalized_base',    'Stage 1 (analytic)'),
    ('normalized',         'Stage 2 (DL + MMD)'),
]

fig, axes = plt.subplots(len(plot_markers), len(layers_to_show),
                          figsize=(6 * len(layers_to_show), 4 * len(plot_markers)))
if len(plot_markers) == 1:
    axes = axes.reshape(1, -1)

for row, mname in enumerate(plot_markers):
    m_idx = marker_names.index(mname)
    for col, (layer_key, title) in enumerate(layers_to_show):
        ax = axes[row, col]
        if layer_key is None:
            data = X_raw
        else:
            data = adata_norm.layers[layer_key]
        for b in unique_batches:
            mask = batch_vals == b
            vals = np.log1p(np.clip(np.asarray(data)[mask, m_idx], 0, None))
            ax.hist(vals, bins=80, alpha=0.4, density=True, label=str(b))
        ax.set_title(f'{mname} — {title}')
        ax.set_xlabel('log1p intensity')
        if col == 0:
            ax.set_ylabel('Density')
        if row == 0 and col == 0:
            ax.legend(fontsize=6, title='Batch', ncol=2)

plt.suptitle('Unimodal Marker Distributions: Raw → Stage 1 → Stage 2\n'
             '(unimodal markers chosen — bimodal markers are masked in Stage 2 MMD and unchanged)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Batch adj-R² Diagnostics

Adjusted R² for each marker regressed on batch labels.
Lower = batch effect better removed.

In [ ]:
batch_arr = adata_norm.obs['batch_id'].values
X_base_arr = np.asarray(adata_norm.layers['normalized_base'])
X_norm_arr = np.asarray(adata_norm.layers['normalized'])

r2_raw   = per_marker_batch_r2(np.log1p(np.clip(X_raw, 0, None)),   batch_arr, marker_names)
r2_base  = per_marker_batch_r2(np.log1p(np.clip(X_base_arr, 0, None)), batch_arr, marker_names)
r2_shift = per_marker_batch_r2(np.log1p(np.clip(X_norm_arr, 0, None)), batch_arr, marker_names)

r2_compare = (
    r2_raw .rename(columns={'adj_r2': 'raw'})
    .merge(r2_base .rename(columns={'adj_r2': 'stage1_base'}), on='marker')
    .merge(r2_shift.rename(columns={'adj_r2': 'stage2_norm'}), on='marker')
)

print('Per-Marker Batch adj-R² (lower = better):\n')
print(r2_compare.to_string(index=False, float_format='%.4f'))
print(f'\nMean adj-R²:')
for col in ['raw', 'stage1_base', 'stage2_norm']:
    print(f'  {col:>14s}: {r2_compare[col].mean():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(r2_compare))
w = 0.25
ax.bar(x - w, r2_compare['raw'],         w, label='Raw',           color='salmon',      alpha=0.85)
ax.bar(x,     r2_compare['stage1_base'], w, label='Stage 1 (base)',color='steelblue',   alpha=0.85)
ax.bar(x + w, r2_compare['stage2_norm'], w, label='Stage 2 (norm)',color='forestgreen', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(r2_compare['marker'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Adjusted R²')
ax.set_title('Per-Marker Batch adj-R² — lower is better')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.5, label='Target (0.05)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Positive Population Preservation Check

% positive cells per marker per sample using **per-sample GMM** (2-component Gaussian mixture
fitted on each sample's raw data in log10 space; threshold applied to both raw and normalized
for that sample). Delta = normalized − raw. Target: |delta| < 5% per marker.

**Headline metric**: the full per-marker mean Δ (± SD) table over all 20 markers, sorted by
mean Δ, with a `pass_5pct` column and a count of markers within ±5%. This is the primary
biology-preservation result. A high SD (CD45, CDX2, DAPI_R1, ...) flags a unimodal marker whose
GMM threshold sits near the histogram peak — its large |Δ| is threshold instability, not real
biology distortion. Read mean Δ alongside its SD; do **not** drop markers.

The density-at-threshold reliability filter (`summarize_positive_population()`) is kept **for
reference only** at the end of the cell — it drops most unimodal markers (often leaving ~3),
too few to compare methods, so it is not the primary metric.

In [ ]:
pop_table_base = positive_population_table(
    adata_norm, norm_layer='normalized_base', sample_col='sample_id'
)
pop_table = positive_population_table(
    adata_norm, norm_layer='normalized', sample_col='sample_id'
)

# ── HEADLINE: full per-marker table, ALL 20 markers (primary biology metric) ──
# mean Δ = mean over samples of (norm − raw) % positive. SD = inter-sample spread.
# High SD (CD45, CDX2, DAPI_R1, ...) flags a unimodal marker whose GMM threshold sits
# near the histogram peak → the large |Δ| reflects threshold instability, NOT real
# biology distortion. Read mean Δ alongside its SD; do not filter markers out.
def pop_summary(pt):
    s = pt.groupby('marker').agg(
        mean_pct_raw=('pct_pos_raw', 'mean'),
        mean_pct_norm=('pct_pos_norm', 'mean'),
        mean_delta=('delta', 'mean'),
        sd_delta=('delta', 'std'),
        mean_abs_delta=('delta', lambda x: x.abs().mean()),
    ).round(2)
    s['pass_5pct'] = s['mean_delta'].abs() < 5
    return s.sort_values('mean_delta')

for label, pt in [('Stage 1 (base)', pop_table_base), ('Stage 2 GNN (norm)', pop_table)]:
    summary = pop_summary(pt)
    n_pass = int(summary['pass_5pct'].sum())
    print(f'Positive Population Preservation — {label} vs raw  (ALL 20 markers):')
    print('=' * 95)
    print(summary.to_string())
    print(f'\n  Markers within ±5% mean Δ: {n_pass}/{len(summary)}')
    print(f'  Overall mean |Δ|: {pt["delta"].abs().mean():.2f}%   max |Δ|: {pt["delta"].abs().max():.2f}%')
    print()

print('Target: |mean Δ| < 5% per marker. High-SD failing markers are unimodal (threshold')
print('near the histogram peak) — interpret their |Δ| as threshold instability, not distortion.')

# ── Secondary (reference only): reliability-filtered view ─────────────────────
# The density-at-threshold filter drops most unimodal markers — often leaving only
# ~3 markers, too few to compare methods. Kept for reference; NOT the headline metric.
print('\n' + '─' * 95)
print('Secondary (reference only) — reliability-filtered; drops most markers, not primary:')
print('─' * 95)
_ = summarize_positive_population(pop_table_base, min_reliable_frac=0.5, label='Stage 1 (base)')
_ = summarize_positive_population(pop_table,      min_reliable_frac=0.5, label='Stage 2 GNN (norm)')


In [ ]:
pivot = pop_table.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: Normalized vs Raw')
plt.tight_layout()
plt.show()

In [ ]:
plot_markers_pop = bimodal_names[:4] if len(bimodal_names) >= 4 else bimodal_names[:2]

fig, axes = plt.subplots(1, max(len(plot_markers_pop), 1),
                          figsize=(5 * max(len(plot_markers_pop), 1), 5))
if len(plot_markers_pop) == 1:
    axes = [axes]

for ax, marker in zip(axes, plot_markers_pop):
    sub = pop_table[pop_table['marker'] == marker]
    ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
    lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x (no change)')
    ax.set_xlabel('% Positive (Raw)')
    ax.set_ylabel('% Positive (Stage 2 normalized)')
    ax.set_title(marker)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Positive Population Preservation (Stage 2 vs Raw)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np, pandas as pd

S2_LABEL = 'Stage 2 GNN'

METHODS_DICT = {'Stage 1': pop_table_base, S2_LABEL: pop_table}

# --- Aggregate mean and SD of delta per marker ---
agg = {}
for method_name, pt in METHODS_DICT.items():
    g = pt.groupby('marker')['delta'].agg(['mean', 'std']).reset_index()
    g.columns = ['marker', 'mean_delta', 'std_delta']
    agg[method_name] = g

order        = list(adata.var_names)
method_names = list(METHODS_DICT.keys())
n_methods    = len(method_names)
n_markers    = len(order)

all_stds   = pd.concat([g['std_delta']        for g in agg.values()])
all_abs    = pd.concat([g['mean_delta'].abs() for g in agg.values()])
vmin, vmax = 0.0, float(all_stds.max())
size_scale = 800.0 / max(float(all_abs.max()), 1.0)
cmap       = plt.cm.coolwarm

fig, ax = plt.subplots(figsize=(2 + n_methods * 2, max(8, n_markers * 0.45)))

for x_idx, method_name in enumerate(method_names):
    g     = agg[method_name].set_index('marker').loc[order].reset_index()
    y_pos = np.arange(n_markers)
    sizes = np.abs(g['mean_delta'].values) * size_scale + 20
    ax.scatter(
        np.full(n_markers, x_idx), y_pos,
        s=sizes, c=g['std_delta'],
        cmap=cmap, vmin=vmin, vmax=vmax,
        edgecolors='k', linewidths=0.4, alpha=0.88,
    )

ax.set_xticks(range(n_methods))
ax.set_xticklabels(method_names, fontsize=11)
ax.set_yticks(range(n_markers))
ax.set_yticklabels(order, fontsize=9)
ax.set_xlim(-0.6, n_methods - 0.4)
ax.set_ylim(-0.7, n_markers - 0.3)
ax.invert_yaxis()
ax.grid(True, alpha=0.2)

for idx, mname in enumerate(order):
    if mname in bimodal_names:
        ax.get_yticklabels()[idx].set_color('navy')
        ax.get_yticklabels()[idx].set_fontweight('bold')

# Size legend
for ref_delta in [5, 10, 20]:
    ax.scatter([], [], s=ref_delta * size_scale + 20, c='gray', alpha=0.7,
               edgecolors='k', linewidths=0.4, label=f'{ref_delta}%')
ax.legend(title='|mean Δ|', loc='lower right', fontsize=8, title_fontsize=8, framealpha=0.9)

plt.colorbar(
    cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap),
    ax=ax,
    label='SD of Δ across samples\n(blue = consistent  ·  red = variable)',
    shrink=0.6,
)
ax.set_title(
    f'Positive Population Change — Stage 1 vs {S2_LABEL}\n'
    'Bubble size ∝ |mean Δ|  ·  Color = inter-sample SD  ·  Target: |Δ| < 5%\n'
    'Navy bold = bimodal markers',
    fontsize=11,
)
plt.tight_layout()
plt.show()

# --- Summary table ---
frames = []
for method_name, pt in METHODS_DICT.items():
    tmp = pt[['marker', 'delta']].copy()
    tmp['method'] = method_name
    frames.append(tmp)
all_delta = pd.concat(frames, ignore_index=True)

pivot = (
    all_delta.groupby(['marker', 'method'])['delta']
    .agg(['mean', 'std']).round(2)
    .unstack('method')
)
print('\nMean Δ (±SD) positive population per marker:')
print('=' * 70)
print(pivot.to_string())
print('\nTarget: |mean Δ| < 5% per marker')

In [ ]:
# === DETAILED PER-SAMPLE BREAKDOWN ===
# Show which samples are driving the large mean Δ values
# Identifies outlier samples and compares median vs mean

print('='*90)
print('DETAILED PER-SAMPLE BREAKDOWN — All Markers')
print('='*90)

S2_LABEL = 'Stage 2 GNN'
pop_tables = {'Stage 1': pop_table_base, S2_LABEL: pop_table}

for method_label, pt in pop_tables.items():
    print(f'\n{method_label} — Per-Sample Positive Population Delta\n')
    print('='*90)
    
    for marker in sorted(pt['marker'].unique()):
        marker_df = pt[pt['marker'] == marker].sort_values('delta')
        
        print(f'\n{marker}:')
        print('-'*90)
        print(marker_df[['sample', 'pct_pos_raw', 'pct_pos_norm', 'delta']].to_string(index=False))
        
        # Statistics
        mean_delta = marker_df['delta'].mean()
        median_delta = marker_df['delta'].median()
        std_delta = marker_df['delta'].std()
        min_delta = marker_df['delta'].min()
        max_delta = marker_df['delta'].max()
        
        within_5 = (abs(marker_df['delta']) <= 5).sum()
        total = len(marker_df)
        
        print(f'\nStats: Mean={mean_delta:+.2f}%, Median={median_delta:+.2f}%, '
              f'Std={std_delta:.2f}%, Range=[{min_delta:+.2f}%, {max_delta:+.2f}%]')
        print(f'Samples within ±5%: {within_5}/{total}')
        
        # Flag outliers
        outliers = marker_df[abs(marker_df['delta']) > 10]
        if len(outliers) > 0:
            print(f'⚠️  OUTLIERS (|Δ| > 10%):')
            for _, row in outliers.iterrows():
                print(f'    {row["sample"]}: {row["delta"]:+.2f}%')
    
    print(f'\n{method_label} SUMMARY:')
    print('-'*90)
    summary = pt.groupby('marker').agg({
        'delta': ['mean', 'median', 'std', 'min', 'max', 
                  lambda x: (abs(x) <= 5).sum()]
    }).round(2)
    summary.columns = ['mean', 'median', 'std', 'min', 'max', 'within_5_count']
    print(summary.to_string())
    
    print(f'\nTarget: |mean Δ| < 5% per marker\n')

## 7. Histogram Comparison

Per-sample histogram PDFs for all markers. Saved to `histograms_shift/`.

In [ ]:
import os
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import colormaps

save_dir = 'histograms_shift'
os.makedirs(save_dir, exist_ok=True)

layers_to_plot = [
    ('normalized_base', 'Stage 1 (analytic)'),
    ('normalized',      'Stage 2 (DL + MMD)'),
]

sample_col = 'sample_id' if 'sample_id' in adata_norm.obs.columns else 'batch_id'
sample_ids = adata_norm.obs[sample_col].astype(str).values
unique_samples = sorted(np.unique(sample_ids).tolist())
cmap = colormaps.get_cmap('tab20')
colors = {s: cmap(i % 20) for i, s in enumerate(unique_samples)}

pdf_path = os.path.join(save_dir, 'shift_histograms.pdf')
rows_per_page = 4

with PdfPages(pdf_path) as pdf:
    page_markers = []
    for i, mname in enumerate(marker_names):
        page_markers.append(mname)
        if len(page_markers) == rows_per_page or i == len(marker_names) - 1:
            fig, axes = plt.subplots(len(page_markers), len(layers_to_plot),
                                     figsize=(7 * len(layers_to_plot), 4 * len(page_markers)))
            if len(page_markers) == 1:
                axes = axes.reshape(1, -1)

            for row, mname in enumerate(page_markers):
                m_idx = marker_names.index(mname)
                for col, (layer_key, label) in enumerate(layers_to_plot):
                    ax = axes[row, col]
                    X_col = np.asarray(adata_norm.layers[layer_key][:, m_idx]).ravel()
                    X_log = np.log1p(np.clip(X_col, 0, None))
                    for s in unique_samples:
                        mask = sample_ids == s
                        c, e = np.histogram(X_log[mask], bins=80)
                        ax.plot(e[:-1], c, color=colors[s], linewidth=1.5, alpha=0.7, label=s)
                    bim_tag = ' [BIMODAL]' if marker_is_bimodal[m_idx] else ''
                    ax.set_title(f'{mname}{bim_tag} — {label}', fontsize=9)
                    ax.set_xlabel('log1p intensity', fontsize=8)
                    if col == 0:
                        ax.set_ylabel('Count', fontsize=8)
                    if row == 0 and col == 0:
                        ax.legend(fontsize=5, ncol=3, frameon=False)
                    ax.spines[['top', 'right']].set_visible(False)

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
            page_markers = []

print(f'PDF saved to: {pdf_path}')

## 8. kBET Evaluation

Compute kBET acceptance rate for 5 clinical groups pairing samples from different batches.

- **Higher kBET = better** (batches well-mixed in local neighborhoods)
- **Lower chi-square = better**

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

EVAL_LAYERS = ['normalized_base', 'normalized']
print(f'Evaluating: {EVAL_LAYERS}')
print()
print('Baselines (g1–g5 mean kBET):')
print('  shift_normalize.py (Stage 1 scipy): 0.6307')
print('  UniFORM:                            0.631')
print('  SpaNCy-GNN ensemble hybrid:         0.574')
print()
print('Stage 1 = analytic (scipy equivalent — expected ~0.631)')
print('Stage 2 = DL + MMD (target > 0.631)')

In [ ]:
all_kbet = {}
umap_coords = {}

for layer_name in EVAL_LAYERS:
    print(f'\n{"-"*55}\n  {layer_name}\n{"-"*55}')
    adata_kbet = adata_norm.copy()
    adata_kbet.X = adata_norm.layers[layer_name].copy()

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}

    all_kbet[layer_name] = layer_res

print('\nDone.')

In [ ]:
print('=' * 70)
print('kBET Summary (mean across groups — higher = better)')
print('=' * 70)

summary_rows = []
for layer_name, res in all_kbet.items():
    kbets = [v['kBET'] for v in res.values() if not np.isnan(v['kBET'])]
    chi2s = [v['chi2'] for v in res.values() if not np.isnan(v['chi2'])]
    if kbets:
        row = {
            'layer': layer_name,
            'mean_kBET': float(np.mean(kbets)),
            'mean_chi2': float(np.mean(chi2s)),
            'n_groups': f'{len(kbets)}/{len(res)}',
        }
        summary_rows.append(row)
        print(f"  {layer_name:>20s}  kBET={row['mean_kBET']:.4f}  "
              f"chi2={row['mean_chi2']:.4f}  ({row['n_groups']} groups)")

print()
for gname, res in all_kbet.get('normalized', {}).items():
    if not np.isnan(res['kBET']):
        print(f"{gname}: kBET={res['kBET']:.4f}  chi2={res['chi2']:.4f}  p={res['p']:.4f}")

print()
print('Baselines:')
print('  Stage 1 expected:           kBET~0.631  (analytic, matches scipy shift_normalize.py)')
print('  UniFORM:                    kBET=0.631')
print(f'  Stage 2 (GNN alpha={HYBRID_ALPHA}):   kBET=???  (target > 0.631)')

df_kbet = pd.DataFrame(summary_rows)

if len(df_kbet) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colors = ['steelblue', 'forestgreen'][:len(df_kbet)]
    axes[0].bar(df_kbet['layer'], df_kbet['mean_kBET'], color=colors, alpha=0.85)
    axes[0].axhline(0.631, color='red', linestyle='--', linewidth=1.5, label='UniFORM 0.631')
    axes[0].set_ylabel('Mean kBET acceptance rate')
    axes[0].set_title('kBET (higher = better)')
    axes[0].set_xticklabels(df_kbet['layer'], rotation=20, ha='right', fontsize=9)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    axes[1].bar(df_kbet['layer'], df_kbet['mean_chi2'], color=colors, alpha=0.85)
    axes[1].set_ylabel('Mean chi-square')
    axes[1].set_title('Chi-square (lower = better)')
    axes[1].set_xticklabels(df_kbet['layer'], rotation=20, ha='right', fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

In [ ]:
# Silhouette score — UniFORM-style: 3 cell types, computed PER SAMPLE
# Tumor epithelial (ECAD+), immune (CD45+), non-immune stromal (aSMA+)
#
# Why per-sample (not per clinical group): computing silhouette across two
# batch-paired samples lets the cross-batch offset inflate the raw baseline —
# good normalization then looks like it *lowers* silhouette (it just removed the
# batch-driven separation). Computing within each sample removes that confound:
# the score reflects only whether the 3 cell populations stay distinct.
# Labels are assigned once on raw data, fixed across all methods.

N_SIL = 3000  # cells per sample; O(n²) — stable well below this

# Step 1: GMM thresholds for 3 anchor markers on full raw dataset (log1p space)
anchor_names = ['ECAD', 'CD45', 'aSMA']
anchor_thresholds = {}
print('Anchor marker GMM thresholds (log1p, raw data):')
for mname in anchor_names:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(
        X_log_preview[:, mi].reshape(-1, 1)
    )
    thr = float(np.mean(gmm.means_))
    anchor_thresholds[mname] = thr
    print(f'  {mname}: {thr:.3f}')

# Step 2: assign cell type labels on raw data — fixed for all methods
#   0 = tumor epithelial (ECAD+)
#   1 = immune           (CD45+)
#   2 = stromal          (aSMA+)
#  -1 = unassigned
# Priority for conflicts: ECAD > CD45 > aSMA
ecad_pos = X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']
cd45_pos = X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']
asma_pos = X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[asma_pos] = 2
cell_type[cd45_pos] = 1
cell_type[ecad_pos] = 0

print('\nCell type assignments (raw data):')
for code, label in [(0, 'tumor epithelial (ECAD+)'),
                    (1, 'immune (CD45+)'),
                    (2, 'stromal (aSMA+)'),
                    (-1, 'unassigned')]:
    n = (cell_type == code).sum()
    print(f'  {label}: {n:,}  ({100*n/len(cell_type):.1f}%)')

# Step 3: silhouette per sample, per method (no cross-batch mixing)
print(f'\nSilhouette score per sample — 3 cell types in 20D log1p space (higher = better):')
sample_ids_all = adata_norm.obs['sample_id'].values
sil_rows = []
for sample in sorted(np.unique(sample_ids_all)):
    smask = sample_ids_all == sample
    s_labels = cell_type[smask]
    valid = np.where(s_labels >= 0)[0]
    if len(valid) < 15:
        continue

    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sel_labels = s_labels[sel]
    classes = np.unique(sel_labels)
    if len(classes) < 2 or any((sel_labels == c).sum() < 5 for c in classes):
        continue  # need ≥2 cell types with ≥5 cells each

    sil_by_layer = {}
    for layer_name in ['raw'] + EVAL_LAYERS:
        if layer_name == 'raw':
            Xs = X_raw[smask]
        else:
            Xs = np.asarray(adata_norm.layers[layer_name])[smask]
        X_log = np.log1p(np.clip(Xs[sel], 0, None))
        sil_by_layer[layer_name] = round(
            float(silhouette_score(X_log, sel_labels, metric='euclidean')), 4
        )
    sil_rows.append({'sample': sample, **sil_by_layer})

sil_df = pd.DataFrame(sil_rows).set_index('sample')
sil_df.loc['Mean'] = sil_df.mean().round(4)
print(sil_df.to_string())
print(f'\n({len(sil_rows)} samples passed the ≥2-cell-type guard)')
print('(higher = better — tumor / immune / stromal populations stay distinct after normalization)')


In [ ]:
VIZ_LAYER = 'normalized'
palette = plt.cm.tab10.colors
fig, axes = plt.subplots(5, 2, figsize=(12, 28))
fig.suptitle('Batch mixing per clinical group — Raw vs Normalized', fontsize=14, y=1.01)

for row_idx, (gname, pairs) in enumerate(GROUP_DEFS.items()):
    parts = [subset_ab(adata_norm, s, b) for s, b in pairs]
    adata_g = parts[0].concatenate(parts[1], batch_key=None)
    unique_b = sorted(adata_g.obs['batch_id'].unique())

    # Raw UMAP — compute via pegasus (same pipeline as kBET)
    umap_raw, batch_raw = None, None
    try:
        mmdata_raw = pgio.MultimodalData(adata_g.copy())
        pg.qc_metrics(mmdata_raw); pg.filter_data(mmdata_raw)
        pg.identify_robust_genes(mmdata_raw)
        pg.pca(mmdata_raw, features=None, n_components=20)
        pg.neighbors(mmdata_raw); pg.umap(mmdata_raw)
        umap_raw  = mmdata_raw.obsm['X_umap']
        batch_raw = mmdata_raw.obs['batch_id'].values
    except Exception as e:
        print(f'{gname} raw UMAP failed: {e}')

    # Normalized UMAP — reuse from kBET loop (no recomputation)
    umap_nrm, batch_nrm = None, None
    if VIZ_LAYER in umap_coords and gname in umap_coords[VIZ_LAYER]:
        umap_nrm, batch_nrm = umap_coords[VIZ_LAYER][gname]

    for col_idx, (coords, batch_ids, title) in enumerate([
            (umap_raw, batch_raw, 'Raw'),
            (umap_nrm, batch_nrm, f'Normalized ({VIZ_LAYER})'),
    ]):
        ax = axes[row_idx, col_idx]
        if coords is None:
            ax.text(0.5, 0.5, 'UMAP unavailable', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{gname} — {title}', fontsize=10)
            continue
        for ci, b in enumerate(unique_b):
            mask = batch_ids == b
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=[palette[ci % len(palette)]], s=2, alpha=0.3, rasterized=True, label=b)
        ax.set_title(f'{gname} — {title}', fontsize=10)
        ax.set_xlabel('UMAP 1', fontsize=8)
        ax.set_ylabel('UMAP 2', fontsize=8)
        ax.tick_params(labelsize=7)
        if col_idx == 1:
            ax.legend(markerscale=4, fontsize=8, loc='upper right',
                      title='batch_id', title_fontsize=8)

plt.tight_layout()
plt.savefig('umap_batch_mixing.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: umap_batch_mixing.png')